<a href="https://colab.research.google.com/github/thechiragbatra/customer-churn-mlops/blob/main/4_streamlit.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [6]:
# ── CELL 1: Install ────────────────────────────────────
!pip install streamlit xgboost causalml shap pyngrok -q

In [7]:
# ── CELL 2: Write Streamlit App ───────────────────────
app_code = '''
import streamlit as st
import pandas as pd
import numpy as np
import pickle
import shap
import matplotlib.pyplot as plt
from xgboost import XGBClassifier

# Page config
st.set_page_config(
    page_title="Churn Prediction",
    page_icon="🔄",
    layout="wide"
)

# Load models
@st.cache_resource
def load_models():
    with open("churn_model.pkl", "rb") as f:
        churn_model = pickle.load(f)
    with open("uplift_model.pkl", "rb") as f:
        uplift_model = pickle.load(f)
    with open("feature_names.pkl", "rb") as f:
        feature_names = pickle.load(f)
    return churn_model, uplift_model, feature_names

churn_model, uplift_model, feature_names = load_models()

# Header
st.title("🔄 Customer Churn Prediction")
st.markdown("**Predicts not just who will churn — but who will respond to a retention offer.**")
st.divider()

# Sidebar inputs
st.sidebar.header("📋 Customer Details")

tenure          = st.sidebar.slider("Tenure (months)", 0, 72, 12)
monthly_charges = st.sidebar.slider("Monthly Charges ($)", 18.0, 120.0, 65.0)
total_charges   = st.sidebar.number_input("Total Charges ($)", 0.0, 9000.0, float(tenure * monthly_charges))
contract        = st.sidebar.selectbox("Contract Type", ["Month-to-month", "One year", "Two year"])
internet        = st.sidebar.selectbox("Internet Service", ["DSL", "Fiber optic", "No"])
senior          = st.sidebar.selectbox("Senior Citizen", ["No", "Yes"])
partner         = st.sidebar.selectbox("Has Partner", ["No", "Yes"])
dependents      = st.sidebar.selectbox("Has Dependents", ["No", "Yes"])
paperless       = st.sidebar.selectbox("Paperless Billing", ["No", "Yes"])

# Encode inputs
contract_map  = {"Month-to-month": 0, "One year": 1, "Two year": 2}
internet_map  = {"DSL": 0, "Fiber optic": 1, "No": 2}
binary_map    = {"No": 0, "Yes": 1}

input_data = np.array([[
    1, binary_map[senior], binary_map[partner], binary_map[dependents],
    tenure, 1, 0, internet_map[internet], 0, 0, 0, 0, 0, 0,
    contract_map[contract], binary_map[paperless], 2,
    monthly_charges, total_charges
]])

# Predict
churn_prob = float(churn_model.predict_proba(input_data)[0][1])
ite        = float(uplift_model.predict(X=input_data).flatten()[0])

# Segment
if churn_prob > 0.5 and ite < -0.05:
    segment = "Persuadable"
    rec     = "🎯 Offer 20% discount immediately"
    color   = "#e74c3c"
elif churn_prob <= 0.5 and ite < -0.05:
    segment = "Sure Thing"
    rec     = "✅ No action needed"
    color   = "#2ecc71"
elif churn_prob > 0.5 and ite >= -0.05:
    segment = "Lost Cause"
    rec     = "❌ Accept churn — offer won\'t help"
    color   = "#e67e22"
else:
    segment = "Sleeping Dog"
    rec     = "⚠️ Do not contact"
    color   = "#95a5a6"

# Results
col1, col2, col3 = st.columns(3)
col1.metric("Churn Probability", f"{churn_prob:.1%}",
            delta="High Risk" if churn_prob > 0.5 else "Low Risk")
col2.metric("Uplift Score", f"{ite:.4f}",
            delta="Responds to offer" if ite < -0.05 else "Won\'t respond")
col3.metric("Segment", segment)

st.divider()

# Recommendation
st.subheader("💡 Recommendation")
st.markdown(f"<div style=\'background:{color}22; border-left:4px solid {color}; padding:16px; border-radius:8px; font-size:18px\'>{rec}</div>", unsafe_allow_html=True)

st.divider()

# SHAP
st.subheader("🔍 Why is this customer at risk?")
explainer  = shap.TreeExplainer(churn_model)
shap_vals  = explainer.shap_values(input_data)[0]
top_idx    = np.argsort(np.abs(shap_vals))[::-1][:8]

fig, ax = plt.subplots(figsize=(8, 4))
colors  = ["#e74c3c" if v > 0 else "#2ecc71" for v in shap_vals[top_idx]]
ax.barh([feature_names[i] for i in top_idx[::-1]],
        shap_vals[top_idx[::-1]], color=colors[::-1])
ax.axvline(0, color="black", linewidth=0.8)
ax.set_xlabel("SHAP Value (impact on churn probability)")
ax.set_title("Feature Impact")
plt.tight_layout()
st.pyplot(fig)

st.divider()
st.caption("Built by Chirag Batra · Customer Churn MLOps Project")
'''

with open("streamlit_app.py", "w") as f:
    f.write(app_code)

print("✅ streamlit_app.py written!")

✅ streamlit_app.py written!


In [ ]:
# ── CELL 3: Upload models + Launch ────────────────────
from google.colab import files
import subprocess
import threading
import time

print("Upload churn_model.pkl, uplift_model.pkl, feature_names.pkl")
uploaded = files.upload()

def run_streamlit():
    subprocess.run([
        "streamlit", "run", "streamlit_app.py",
        "--server.port", "8501",
        "--server.headless", "true"
    ])

t = threading.Thread(target=run_streamlit, daemon=True)
t.start()
time.sleep(5)

from pyngrok import ngrok
public_url = ngrok.connect(8501)
print(f"\n🚀 Dashboard is live at: {public_url}")

Upload churn_model.pkl, uplift_model.pkl, feature_names.pkl


In [ ]:
from pyngrok import ngrok

ngrok.set_auth_token("3E8ZxzxyrO3zyjep0Pi73HjoFpI_3ZySmnyFNfkgwzm8ngZGQ")

public_url = ngrok.connect(8501)
print(f"\n🚀 Dashboard is live at: {public_url}")

In [ ]:
app_code = '''
import streamlit as st
import pandas as pd
import numpy as np
import pickle
import shap
import matplotlib.pyplot as plt
import matplotlib
matplotlib.rcParams["figure.facecolor"] = "#0e1117"
matplotlib.rcParams["axes.facecolor"] = "#0e1117"
matplotlib.rcParams["text.color"] = "white"
matplotlib.rcParams["axes.labelcolor"] = "white"
matplotlib.rcParams["xtick.color"] = "white"
matplotlib.rcParams["ytick.color"] = "white"

st.set_page_config(
    page_title="Churn Intelligence",
    page_icon="🧠",
    layout="wide"
)

# ── Custom CSS ──────────────────────────────────────────
st.markdown("""
<style>
@import url('https://fonts.googleapis.com/css2?family=Inter:wght@300;400;600;700&display=swap');

html, body, [class*="css"] { font-family: 'Inter', sans-serif; }

.main { background: #0e1117; }

.metric-card {
    background: linear-gradient(135deg, #1a1d2e 0%, #16213e 100%);
    border: 1px solid #2d3561;
    border-radius: 16px;
    padding: 24px;
    text-align: center;
    box-shadow: 0 4px 24px rgba(0,0,0,0.3);
}
.metric-value { font-size: 2.4rem; font-weight: 700; margin: 8px 0; }
.metric-label { font-size: 0.85rem; color: #8892b0; text-transform: uppercase; letter-spacing: 1px; }
.metric-badge {
    display: inline-block;
    padding: 4px 12px;
    border-radius: 20px;
    font-size: 0.75rem;
    font-weight: 600;
    margin-top: 8px;
}

.rec-card {
    border-radius: 16px;
    padding: 20px 28px;
    font-size: 1.1rem;
    font-weight: 600;
    margin: 8px 0;
    border-left: 5px solid;
}

.section-title {
    font-size: 1.1rem;
    font-weight: 600;
    color: #ccd6f6;
    text-transform: uppercase;
    letter-spacing: 2px;
    margin-bottom: 16px;
}

.sidebar-section {
    background: #1a1d2e;
    border-radius: 12px;
    padding: 16px;
    margin-bottom: 12px;
}

div[data-testid="stSidebar"] {
    background: #0d1117;
    border-right: 1px solid #21262d;
}

.stSlider > div > div > div > div { background: #4f8ef7 !important; }
</style>
""", unsafe_allow_html=True)

# ── Load Models ─────────────────────────────────────────
@st.cache_resource
def load_models():
    with open("churn_model.pkl", "rb") as f:
        churn_model = pickle.load(f)
    with open("uplift_model.pkl", "rb") as f:
        uplift_model = pickle.load(f)
    with open("feature_names.pkl", "rb") as f:
        feature_names = pickle.load(f)
    return churn_model, uplift_model, feature_names

churn_model, uplift_model, feature_names = load_models()

# ── Sidebar ─────────────────────────────────────────────
with st.sidebar:
    st.markdown("## 🧠 Churn Intelligence")
    st.markdown("---")
    st.markdown("### 📋 Customer Profile")

    tenure          = st.slider("Tenure (months)", 0, 72, 12)
    monthly_charges = st.slider("Monthly Charges ($)", 18.0, 120.0, 65.0)
    total_charges   = st.number_input("Total Charges ($)", 0.0, 9000.0,
                                       float(tenure * monthly_charges))

    st.markdown("---")
    st.markdown("### 📦 Service Details")

    contract  = st.selectbox("Contract Type",
                              ["Month-to-month", "One year", "Two year"])
    internet  = st.selectbox("Internet Service",
                              ["DSL", "Fiber optic", "No"])
    payment   = st.selectbox("Payment Method",
                              ["Electronic check", "Mailed check",
                               "Bank transfer", "Credit card"])

    st.markdown("---")
    st.markdown("### 👤 Demographics")

    col1, col2 = st.columns(2)
    senior     = col1.selectbox("Senior", ["No", "Yes"])
    partner    = col2.selectbox("Partner", ["No", "Yes"])
    dependents = col1.selectbox("Dependents", ["No", "Yes"])
    paperless  = col2.selectbox("Paperless", ["No", "Yes"])

# ── Encode ──────────────────────────────────────────────
binary_map   = {"No": 0, "Yes": 1}
contract_map = {"Month-to-month": 0, "One year": 1, "Two year": 2}
internet_map = {"DSL": 0, "Fiber optic": 1, "No": 2}
payment_map  = {"Electronic check": 0, "Mailed check": 1,
                "Bank transfer": 2, "Credit card": 3}

input_data = np.array([[
    1, binary_map[senior], binary_map[partner], binary_map[dependents],
    tenure, 1, 0, internet_map[internet], 0, 0, 0, 0, 0, 0,
    contract_map[contract], binary_map[paperless],
    payment_map[payment], monthly_charges, total_charges
]])

churn_prob = float(churn_model.predict_proba(input_data)[0][1])
ite        = float(uplift_model.predict(X=input_data).flatten()[0])

# Segment logic
if churn_prob > 0.5 and ite < -0.05:
    segment   = "Persuadable"
    rec       = "🎯 Offer 20% discount immediately"
    rec_color = "#e74c3c"
    seg_color = "#ff6b6b"
    seg_bg    = "rgba(231,76,60,0.15)"
elif churn_prob <= 0.5 and ite < -0.05:
    segment   = "Sure Thing"
    rec       = "✅ No action needed — customer is loyal"
    rec_color = "#2ecc71"
    seg_color = "#51cf66"
    seg_bg    = "rgba(46,204,113,0.15)"
elif churn_prob > 0.5 and ite >= -0.05:
    segment   = "Lost Cause"
    rec       = "❌ Accept churn — retention offer won\'t help"
    rec_color = "#e67e22"
    seg_color = "#ffa94d"
    seg_bg    = "rgba(230,126,34,0.15)"
else:
    segment   = "Sleeping Dog"
    rec       = "⚠️ Do not contact — offer may trigger churn"
    rec_color = "#95a5a6"
    seg_color = "#adb5bd"
    seg_bg    = "rgba(149,165,166,0.15)"

# ── Main Header ─────────────────────────────────────────
st.markdown("""
<div style="padding: 8px 0 24px 0">
    <h1 style="font-size:2rem; font-weight:700; color:#ccd6f6; margin:0">
        🧠 Customer Churn Intelligence
    </h1>
    <p style="color:#8892b0; margin:4px 0 0 0; font-size:1rem">
        Predicts not just <b>who</b> will churn — but <b>who responds</b> to a retention offer
    </p>
</div>
""", unsafe_allow_html=True)

# ── Metric Cards ────────────────────────────────────────
c1, c2, c3 = st.columns(3)

risk_color = "#ff6b6b" if churn_prob > 0.5 else "#51cf66"
risk_label = "HIGH RISK" if churn_prob > 0.5 else "LOW RISK"

with c1:
    st.markdown(f"""
    <div class="metric-card">
        <div class="metric-label">Churn Probability</div>
        <div class="metric-value" style="color:{risk_color}">{churn_prob:.1%}</div>
        <span class="metric-badge" style="background:{risk_color}22;color:{risk_color}">{risk_label}</span>
    </div>""", unsafe_allow_html=True)

ite_color = "#51cf66" if ite < -0.05 else "#ffa94d"
ite_label = "RESPONDS TO OFFER" if ite < -0.05 else "WON\'T RESPOND"

with c2:
    st.markdown(f"""
    <div class="metric-card">
        <div class="metric-label">Uplift Score</div>
        <div class="metric-value" style="color:{ite_color}">{ite:.4f}</div>
        <span class="metric-badge" style="background:{ite_color}22;color:{ite_color}">{ite_label}</span>
    </div>""", unsafe_allow_html=True)

with c3:
    st.markdown(f"""
    <div class="metric-card">
        <div class="metric-label">Customer Segment</div>
        <div class="metric-value" style="color:{seg_color};font-size:1.8rem">{segment}</div>
        <span class="metric-badge" style="background:{seg_bg};color:{seg_color}">CAUSAL ML</span>
    </div>""", unsafe_allow_html=True)

st.markdown("<br>", unsafe_allow_html=True)

# ── Recommendation ──────────────────────────────────────
st.markdown(f"""
<div class="rec-card" style="background:{seg_bg};border-color:{rec_color};color:{rec_color}">
    💡 Recommendation &nbsp;·&nbsp; <span style="color:white">{rec}</span>
</div>""", unsafe_allow_html=True)

st.markdown("<br>", unsafe_allow_html=True)

# ── SHAP Chart ──────────────────────────────────────────
st.markdown('<div class="section-title">🔍 Why is this customer at risk?</div>',
            unsafe_allow_html=True)

explainer = shap.TreeExplainer(churn_model)
shap_vals = explainer.shap_values(input_data)[0]
top_idx   = np.argsort(np.abs(shap_vals))[::-1][:8]

fig, ax = plt.subplots(figsize=(9, 4))
colors  = ["#ff6b6b" if v > 0 else "#51cf66" for v in shap_vals[top_idx[::-1]]]
bars    = ax.barh([feature_names[i] for i in top_idx[::-1]],
                   shap_vals[top_idx[::-1]], color=colors, height=0.6)
ax.axvline(0, color="#4a5568", linewidth=1, linestyle="--")
ax.set_xlabel("SHAP Value  (→ increases churn  |  ← decreases churn)",
              fontsize=10)
ax.spines["top"].set_visible(False)
ax.spines["right"].set_visible(False)
ax.spines["left"].set_color("#2d3561")
ax.spines["bottom"].set_color("#2d3561")
plt.tight_layout()
st.pyplot(fig)

st.markdown("---")

# ── Segment Guide ───────────────────────────────────────
st.markdown('<div class="section-title">📊 Segment Reference</div>',
            unsafe_allow_html=True)

seg1, seg2, seg3, seg4 = st.columns(4)
for col, s, c, desc in zip(
    [seg1, seg2, seg3, seg4],
    ["✅ Sure Thing", "🎯 Persuadable", "❌ Lost Cause", "⚠️ Sleeping Dog"],
    ["#51cf66", "#ff6b6b", "#ffa94d", "#adb5bd"],
    ["Low risk — loyal customer",
     "High risk + responds to offer",
     "High risk — offer won\'t help",
     "Low risk — don\'t disturb"]
):
    col.markdown(f"""
    <div style="background:{c}18;border:1px solid {c}44;border-radius:12px;
                padding:14px;text-align:center">
        <div style="color:{c};font-weight:700;font-size:0.9rem">{s}</div>
        <div style="color:#8892b0;font-size:0.78rem;margin-top:6px">{desc}</div>
    </div>""", unsafe_allow_html=True)

st.markdown("<br>", unsafe_allow_html=True)
st.caption("🧠 Churn Intelligence · Built by Chirag Batra · Customer Churn MLOps Project")
'''

with open("streamlit_app.py", "w") as f:
    f.write(app_code)

print("✅ Beautiful app written! Streamlit will auto-reload.")